# MADKE-X on free GPUs (Kaggle 2x T4 / Google Colab T4)

This notebook reproduces **Table 10** of *MADKE-X: Traceable, Evidence-Governed Multi-Agent Debate for Knowledge-Intensive QA*
with **no paid API key**:

1. installs a local OpenAI-compatible LLM server (**vLLM**, or **Ollama** as a lighter fallback) and loads a Qwen chat model,
2. runs `run_experiments.py --mode live`, which executes both the unpatched **MADKE (re-run)** baseline and **MADKE-X** on the
   five datasets (TriviaQA, NQ, HotpotQA, 2WikiMultiHopQA, FEVER) with **per-question checkpointing** (re-running the cell resumes),
3. computes F1 / EM / `Cons` / `Co2` / support rate / paired McNemar *p*-values automatically and exports `TABLE10.md`.

| Platform | GPUs | Preset model | Notes |
|---|---|---|---|
| Kaggle (Accelerator = *GPU T4 x2*) | 2 x T4 (2 x 15 GB) | `Qwen/Qwen1.5-32B-Chat-AWQ` (tensor-parallel 2) | the model used in the original MADKE paper |
| Google Colab (free) | 1 x T4 (15 GB) | `Qwen/Qwen2.5-7B-Instruct-AWQ` | fp16 7B weights alone are ~15 GB, so the 4-bit AWQ build is required on one T4 |

> **Before you start (Kaggle):** right panel -> *Session options* -> **Accelerator = GPU T4 x2** and **Internet = On** (phone-verified account).
> **Colab:** *Runtime -> Change runtime type -> T4 GPU* (or L4 / A100 if your plan offers it).
> Run the cells **in order**. Cell 7 first tests the model server ("PREFLIGHT OK") and refuses to start otherwise.
>
> **Time budget.** One question costs ~10-16 LLM calls per system. A 7B AWQ model on one T4 handles roughly 1,000-1,500 calls/hour
> with `CONCURRENCY=8`, so start with `END_INDEX = 100` per dataset (about 1.5-2.5 h for both systems x 5 datasets) and raise it in
> later sessions - checkpoints make every re-run incremental. Kaggle sessions are limited to 12 h (and 30 GPU-hours/week).

In [ ]:
# ============ 1. Configuration ============
import os, subprocess, json, shutil

REPO_URL   = os.environ.get("MADKE_REPO_URL", "")   # e.g. "https://github.com/<user>/MADKE.git"; leave "" if the repo is uploaded as a Kaggle dataset / zip
REPO_DIR   = "/kaggle/working/MADKE" if os.path.isdir("/kaggle/working") else "/content/MADKE"
BACKEND    = "vllm"          # "vllm" (recommended) or "ollama" (lighter, CPU/GPU offload, slower)
DATASETS   = ["triviaqa", "nq", "hotpotqa", "wikimultihopqa", "fever"]
START_INDEX, END_INDEX = 0, 2     # PILOT FIRST: 2 questions. Check real answers in cell 8, then set 100 (or 500) and re-run cell 7
CONCURRENCY = 8                   # parallel questions sent to the server (vLLM batches them)
MAX_TOKENS  = 512
TEMPERATURE = 0.5

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().splitlines()
N_GPU = len(gpus)
print("GPUs:", gpus)
if BACKEND == "vllm":
    if N_GPU >= 2:
        MODEL, TP = "Qwen/Qwen1.5-32B-Chat-AWQ", 2          # Kaggle 2x T4
    else:
        MODEL, TP = "Qwen/Qwen2.5-7B-Instruct-AWQ", 1        # Colab 1x T4 (fp16 7B does not fit with KV cache)
    BASE_URL = "http://localhost:8000/v1"
else:
    MODEL, TP = "qwen2.5:7b-instruct-q4_K_M", 1
    BASE_URL = "http://localhost:11434/v1"
MODEL = os.environ.get("MADKE_MODEL", MODEL)   # override if you want another model
print("Backend:", BACKEND, "| model:", MODEL, "| tensor-parallel:", TP, "| base_url:", BASE_URL)

In [ ]:
# ============ 2. Get the repository ============
import glob
if not os.path.isdir(REPO_DIR):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        # look for an uploaded copy: Kaggle dataset (/kaggle/input/**/run_experiments.py) or a zip in the working dir
        hits = glob.glob("/kaggle/input/**/run_experiments.py", recursive=True) + glob.glob("/content/**/run_experiments.py", recursive=True)
        if hits:
            shutil.copytree(os.path.dirname(hits[0]), REPO_DIR)
        else:
            zips = glob.glob("/kaggle/working/*.zip") + glob.glob("/content/*.zip")
            assert zips, "Set REPO_URL, attach the repository as a Kaggle dataset, or upload a zip of the repo."
            shutil.unpack_archive(zips[0], REPO_DIR)
            inner = glob.glob(REPO_DIR + "/**/run_experiments.py", recursive=True)
            if inner and os.path.dirname(inner[0]) != REPO_DIR:
                for item in os.listdir(os.path.dirname(inner[0])):
                    shutil.move(os.path.join(os.path.dirname(inner[0]), item), REPO_DIR)
os.chdir(REPO_DIR)
print("Repository:", os.getcwd())
print(subprocess.run(["sha256sum"] + sorted(glob.glob("data/*/*.json")), capture_output=True, text=True).stdout)

In [ ]:
# ============ 3. Offline sanity check (no model needed, ~1 min) ============
# Verifies data checksums, runs the scripted pipeline and all invariants on a small slice.
!python run_experiments.py --mode offline --datasets triviaqa fever --end_index 50 --subset 0 --fail_demo 20 --n_boot 200 --out_dir results/offline_smoke

In [ ]:
# ============ 4. Install the model server ============
import sys
if BACKEND == "vllm":
    # vLLM 0.6.x supports T4 (compute capability 7.5), AWQ and tensor-parallel; wheels are abi3 (any Python >= 3.8)
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.6.6.post1"], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    assert r.returncode == 0, "vLLM installation failed"
    os.environ["VLLM_USE_V1"] = "0"
else:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
print("install done")

In [ ]:
# ============ 5. Launch the server in the background and wait until it is ready ============
import time, urllib.request
LOG = "/kaggle/working/server.log" if os.path.isdir("/kaggle/working") else "/content/server.log"
if BACKEND == "vllm":
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL, "--served-model-name", MODEL,
           "--tensor-parallel-size", str(TP), "--dtype", "half", "--max-model-len", "4096",
           "--gpu-memory-utilization", "0.95", "--enforce-eager", "--port", "8000", "--disable-log-requests"]
    if "AWQ" in MODEL.upper():
        cmd += ["--quantization", "awq"]
    server = subprocess.Popen([sys.executable, "-m"] + cmd[2:], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
    health = BASE_URL + "/models"
else:
    server = subprocess.Popen(["ollama", "serve"], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    subprocess.run(["ollama", "pull", MODEL], check=True)
    health = "http://localhost:11434/api/tags"

deadline = time.time() + 40 * 60   # model download + load can take a while (32B-AWQ is ~19 GB)
while time.time() < deadline:
    try:
        urllib.request.urlopen(health, timeout=5).read()
        print("server ready:", health); break
    except Exception:
        if server.poll() is not None:
            print(open(LOG).read()[-4000:]); raise SystemExit("server exited - see log above")
        time.sleep(15)
else:
    print(open(LOG).read()[-4000:]); raise SystemExit("server did not become ready in time")

In [ ]:
# ============ 6. Smoke test one chat completion through the same client the experiments use ============
import sys; sys.path.insert(0, os.getcwd())
from models.openai_compat_agent import OpenAICompatAgent
agent = OpenAICompatAgent({"base_url": BASE_URL, "api_key": "EMPTY", "model": MODEL, "temperature": 0.0, "max_tokens": 64, "max_retries": 2}, "Smoke")
res = agent([{"role": "user", "content": "Answer in the form [answer]: In which city were Rotary Clubs set up in 1905?"}])
print(res)
assert res.ok, res.error

In [ ]:
# ============ 7. Run the live experiments (both systems, 5 datasets) - RESUMABLE ============
# Interrupted? Just re-run this cell: finished questions are read back from results/live/<model>/<dataset>/*.jsonl.
cmd = ["python", "run_experiments.py", "--mode", "live", "--base_url", BASE_URL, "--api_key", "EMPTY", "--model", MODEL,
       "--datasets", *DATASETS, "--start_index", str(START_INDEX), "--end_index", str(END_INDEX),
       "--concurrency", str(CONCURRENCY), "--max_tokens", str(MAX_TOKENS), "--temperature", str(TEMPERATURE),
       "--timeout", "300", "--max_retries", "3"]
print(" ".join(cmd))
!{" ".join(cmd)}

In [ ]:
# ============ 8. Table 10 ============
from IPython.display import Markdown, display
import glob
table = sorted(glob.glob("results/live/*/TABLE10.md"))[-1]
display(Markdown(open(table).read()))

In [ ]:
# ============ 8b. Look at real answers (sanity check of the pilot) ============
import glob, json
for f in sorted(glob.glob("results/live/*/triviaqa/madke_x.jsonl"))[-1:]:
    for line in open(f):
        r = json.loads(line)
        print(r["status"], "| gold:", r["answer"][:2], "| model answer:", r["final_answer"], "| support:", r["trace"]["support_status"])

In [ ]:
# ============ 9. Export results (checkpoints, per-question traces, Table 10) ============
out = "/kaggle/working/madke_x_results" if os.path.isdir("/kaggle/working") else "/content/madke_x_results"
shutil.make_archive(out, "zip", "results")
print("archive:", out + ".zip")
try:
    from google.colab import files   # Colab: trigger a browser download
    files.download(out + ".zip")
except Exception:
    print("On Kaggle the archive is available under the Output tab of the notebook.")

## Notes

* **Resuming.** Every completed question is appended to `results/live/<model>/<dataset>/<system>.jsonl`. Re-running cell 7 skips them;
  add `--retry_failed` to re-run questions whose status is `failed` (e.g. after a server restart).
* **Failure-aware scoring.** Questions whose LLM calls fail after the bounded retries are marked `failed` and excluded from F1/EM/Cons/Co2
  rather than being scored as wrong answers (the `Failed` column of Table 10 reports how many).
* **Other servers.** Any OpenAI-compatible endpoint works: `--base_url https://host/v1 --api_key <key> --model <name>`; use `--ca_bundle`
  for a private CA or `--insecure` for a self-signed local server.
* **Evaluation only.** `python evaluation.py --prediction_path results/live/<model>/<dataset>/madke_x.json --compare_path .../madke_legacy.json`
  recomputes all metrics (no GPT-4 grading needed).